# 03 — (Optionnel) Fine-tuning LoRA / QLoRA

**EFREI 2025-2026 · Solution Delivery · Filière Data — Niveau COULD**

> **Position non clinique.** Ce notebook est pédagogique et expérimental.
> Ne jamais utiliser un modèle fine-tuné ici en conditions réelles.

## Objectif

Ce notebook présente deux pistes de fine-tuning expérimental (**COULD**) :

| Piste | Modèle | Méthode | Prérequis |
|---|---|---|---|
| A | Gemma 4 (Unsloth) | LoRA | GPU, unsloth, transformers |
| B | MedGemma 4B | PEFT/QLoRA | GPU, peft, bitsandbytes, accès HF |

Ces stubs sont **non exécutables** sans GPU et sans les accès requis.
Ils servent à montrer l'architecture de la solution et les choix techniques.

> **Prérequis MUST complétés avant d'explorer cette piste :**
> 1. Baseline reproductible validée (notebook 01)
> 2. Comparaison prompt amélioré documentée (notebook 02)
> 3. 8 tests smoke passés

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

# Vérifier que la baseline est disponible
baseline_path = ROOT / 'eval' / 'outputs' / 'baseline_metrics.json'
if baseline_path.exists():
    print('Baseline reference :', json.loads(baseline_path.read_text()))
else:
    print('Baseline non générée. Exécuter : python eval/run_evaluation.py --mode toy')

# Vérifier GPU
try:
    import torch
    print('\nPyTorch :', torch.__version__)
    print('CUDA disponible :', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU :', torch.cuda.get_device_name(0))
except ImportError:
    print('\nPyTorch non installé — stubs non exécutables sans GPU.')

## Piste A — Gemma 4 avec Unsloth LoRA

**Source :** [Unsloth Gemma 4 Guide](https://unsloth.ai/docs/models/gemma-4/train)

**Conditions d'accès :** Gemma 4 nécessite d'accepter les conditions d'utilisation sur Hugging Face.

**Installation requise :**
```bash
pip install unsloth transformers accelerate datasets
```

In [ ]:
# STUB — non exécutable sans GPU et sans accès HuggingFace
# Voir finetuning/gemma4_unsloth_lora_stub.py pour le code complet

try:
    import torch
    from unsloth import FastVisionModel
    GPU_OK = torch.cuda.is_available()
except ImportError:
    GPU_OK = False
    print('unsloth non disponible — affichage du pseudo-code uniquement.')

if not GPU_OK:
    print("""
--- PSEUDO-CODE (non exécuté) ---

model, tokenizer = FastVisionModel.from_pretrained(
    model_name    = 'unsloth/gemma-4-12b-it',
    load_in_4bit  = True,
    max_seq_length = 2048,
)

model = FastVisionModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout   = 0.05,
)

# Dataset synthétique au format instruction
# {image: ..., question: 'Analyse cette radiographie', answer: <JSON selon schéma>}

trainer = SFTTrainer(model=model, dataset=train_dataset, ...)
trainer.train()
model.save_pretrained('finetuning/gemma4_lora_adapter')
""")

## Piste B — MedGemma 4B avec PEFT/QLoRA

**Source :** [MedGemma Model Card](https://huggingface.co/google/medgemma-4b-pt)

**Conditions d'accès :** MedGemma nécessite d'accepter les Health AI Developer Foundations Terms sur Hugging Face.

**Installation requise :**
```bash
pip install peft bitsandbytes transformers accelerate
```

In [ ]:
# STUB — non exécutable sans GPU et sans accès MedGemma
# Voir finetuning/medgemma_peft_qlora_stub.py pour le code complet

try:
    import torch
    from peft import LoraConfig, get_peft_model
    GPU_OK = torch.cuda.is_available()
except ImportError:
    GPU_OK = False
    print('peft non disponible — affichage du pseudo-code uniquement.')

if not GPU_OK:
    print("""
--- PSEUDO-CODE (non exécuté) ---

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = 'bfloat16',
)

model = AutoModelForCausalLM.from_pretrained(
    'google/medgemma-4b-pt',
    quantization_config = bnb_config,
    device_map          = 'auto',
)

lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout   = 0.05,
    task_type      = 'CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # ~ 0.1% des paramètres
""")

## Protocole d'évaluation post fine-tuning

Si le fine-tuning est réalisé, comparer au même protocole que la baseline :

In [ ]:
# Exemple de tableau de comparaison (métriques fictives)
import pandas as pd

baseline_path = ROOT / 'eval' / 'outputs' / 'baseline_metrics.json'

if baseline_path.exists():
    real_baseline = json.loads(baseline_path.read_text())
else:
    real_baseline = {'accuracy': 1.0, 'macro_f1': 1.0, 'note': 'toy predictor'}

# Métriques fictives d'un fine-tuned (à remplacer par les vraies)
mock_finetuned = {
    'accuracy': 0.87,
    'macro_f1': 0.85,
    'json_valid_rate': 1.0,
    'warning_rate': 1.0,
    'uncertain_rate': 0.20,
}

KEYS = ['accuracy', 'macro_f1', 'json_valid_rate', 'warning_rate', 'uncertain_rate']
comparison = pd.DataFrame([
    {'mode': 'toy_baseline', **{k: real_baseline.get(k, '—') for k in KEYS}},
    {'mode': 'finetuned_mock', **{k: mock_finetuned.get(k, '—') for k in KEYS}},
])
display(comparison)
print('\n(Remplacer finetuned_mock par les vraies métriques après entraînement.)')

## Obligations de citation

| Ressource | Version | Licence / Conditions d'accès |
|---|---|---|
| Unsloth Gemma 4 | catalogue Unsloth | Gemma Terms of Use (Google) |
| MedGemma 4B | `google/medgemma-4b-pt` | Health AI Developer Foundations Terms |
| MIMIC-CXR | 2.1.0 | PhysioNet Credentialed Access (accès contrôlé, non redistribuable) |
| CheXpert | v1.0 | Stanford AIMI — usage non commercial |

> **Rappel :** Ne jamais commiter de données patient réelles, identifiantes ou ambiguës.
> Ce prototype ne doit pas être présenté comme validé médicalement.